In [1]:
!pip install transformers datasets accelerate bitsandbytes peft
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install scikit-learn imbalanced-learn pandas numpy matplotlib seaborn
!pip install onnxruntime-gpu wandb plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 38.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [2]:
import torch
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from imblearn.over_sampling import SMOTE
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                         TrainingArguments, Trainer, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, TaskType
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU Available: True
GPU Name: Tesla T4


In [3]:
log_data = []
with open('/content/drive/MyDrive/subset/HDFS_subset.log', 'r') as f:
    for line in f:
        pattern = r'(\d{6}\s\d{6})\s(\d+)\s(\w+)\s([^:]+):\s(.+)'
        match = re.match(pattern, line.strip())
        if match:
            timestamp, thread, level, component, message = match.groups()

            block_match = re.search(r'(blk_-?\d+)', message)
            block_id = block_match.group(1) if block_match else None

            log_data.append({
                'timestamp': timestamp,
                'thread': thread,
                'level': level,
                'component': component,
                'message': message,
                'block_id': block_id,
                'full_log': line.strip()
            })

logs_df = pd.DataFrame(log_data)
print(f"Total logs parsed: {len(logs_df)}")

labels_df = pd.read_csv('/content/drive/MyDrive/subset/anomaly_label_subset.csv')
labels_df.columns = ['block_id', 'label']

merged_df = logs_df.merge(labels_df, on='block_id', how='inner')
merged_df['label_num'] = (merged_df['label'] == 'Anomaly').astype(int)

print(f"Normal logs: {(merged_df['label_num'] == 0).sum()}")
print(f"Anomaly logs: {(merged_df['label_num'] == 1).sum()}")

Total logs parsed: 1117995
Normal logs: 1089673
Anomaly logs: 28322


In [4]:
merged_df['log_length'] = merged_df['message'].str.len()
merged_df['has_ip'] = merged_df['message'].str.contains(r'\d+\.\d+\.\d+\.\d+').astype(int)
merged_df['has_error'] = merged_df['message'].str.contains('error|Error|ERROR', case=False).astype(int)
merged_df['component_type'] = merged_df['component'].str.split('.').str[-1]

merged_df['model_input'] = merged_df.apply(lambda x:
    f"Component: {x['component_type']} Level: {x['level']} Message: {x['message'][:200]}", axis=1)

X = merged_df['model_input'].values
y = merged_df['label_num'].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=42)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Train anomaly ratio: {y_train.sum() / len(y_train):.3f}")

Train: 880420, Val: 125775, Test: 111800
Train anomaly ratio: 0.025


In [6]:
feature_vectorizer = merged_df[['log_length', 'has_ip', 'has_error']].iloc[:len(X_train)].values

smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_features_balanced, y_train_balanced = smote.fit_resample(feature_vectorizer, y_train)

balanced_indices = []
normal_indices = np.where(y_train == 0)[0]
anomaly_indices = np.where(y_train == 1)[0]

balanced_indices.extend(range(len(y_train)))

n_synthetic = len(y_train_balanced) - len(y_train)
synthetic_indices = np.random.choice(anomaly_indices, n_synthetic, replace=True)
balanced_indices.extend(synthetic_indices)

X_train_balanced = np.concatenate([X_train, X_train[synthetic_indices]])
y_train_balanced = np.concatenate([y_train, y_train[synthetic_indices]])

print(f"Balanced train set: {len(y_train_balanced)} samples")
print(f"Normal: {(y_train_balanced == 0).sum()}, Anomaly: {(y_train_balanced == 1).sum()}")

Balanced train set: 1115550 samples
Normal: 858116, Anomaly: 257434


In [9]:
from torch.utils.data import Dataset

class LogDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

model_name = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = LogDataset(X_train_balanced, y_train_balanced, tokenizer)
val_dataset = LogDataset(X_val, y_val, tokenizer)

print("Datasets created successfully")

Datasets created successfully


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

trainable params: 120,434 || all params: 14,471,308 || trainable%: 0.8322


In [11]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/Research_Stuff/SLM/slm_results/',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='/content/drive/MyDrive/Research_Stuff/SLM/logs/',
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    report_to="none",
    fp16=True,
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [12]:
print("Starting SLM training...")
trainer.train()

trainer.save_model("/content/drive/MyDrive/Research_Stuff/SLM/slm_final")
print("SLM training completed!")

Starting SLM training...


Step,Training Loss,Validation Loss,F1
500,0.563600,0.383504,0.962166
1000,0.533000,0.300200,0.962166
1500,0.527400,0.285892,0.962166
2000,0.536400,0.273304,0.962166
2500,0.525700,0.276028,0.964068
3000,0.539900,0.277117,0.964068
3500,0.527300,0.250294,0.964068
4000,0.515600,0.265661,0.966584
4500,0.513400,0.245262,0.966635
5000,0.506900,0.247149,0.967042


KeyboardInterrupt: 